In [ ]:
import base64
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from pypdf import PdfReader
import re

load_dotenv()


MAX_TOKENS_TOC = 32000

MODELOS = {
    "anthropic": {
        "sonnet": "claude-sonnet-4-5",
        "haiku":  "claude-haiku-4-5",
    },
    "google": {
        "flash":      "gemini-2.5-flash",
        "flash-lite": "gemini-3.1-flash-lite-preview",
    },
}

PROMPT = (
    "Leia o documento COMPLETO, do início ao fim, analisando TODAS as páginas. "
    "Extraia somente a estrutura útil do documento, em ordem, com número de página.\n\n"

    "Inclua apenas itens que sejam cabeçalhos reais de seção.\n\n"

    "Podem ser incluídos:\n"
    "- nível 1 numerado: 1, 2, 3...\n"
    "- nível 2 numerado: 1.1, 1.2, 2.1...\n"
    "- nível 3 numerado: 1.1.1, 1.1.2...\n"
    "- nível 4 numerado: 1.1.1.1, 1.1.1.2...\n"
    "- ANEXOS e APÊNDICES\n"
    "- subtítulos internos de ANEXOS e APÊNDICES até 4 níveis, mesmo sem numeração, desde que sejam cabeçalhos curtos e estruturais\n\n"

    "Distinção entre cabeçalho e parágrafo numerado (CRÍTICO):\n"
    "- muitos documentos usam numeração hierárquica (1, 1.1, 1.1.1) em parágrafos de texto corrido, não em cabeçalhos\n"
    "- numeração NÃO é indício suficiente de cabeçalho\n"
    "- um item só é cabeçalho quando tem rótulo curto e estrutural, visualmente destacado, funcionando como título de seção\n"
    "- se após o número vier texto corrido, frase completa, ou conteúdo que continua parágrafo, NÃO é cabeçalho — OMITA\n"
    "- se houver qualquer dúvida entre cabeçalho e parágrafo, OMITA\n"
    "- a mesma regra vale em qualquer nível (1, 1.1, 1.1.1, 1.1.1.1) e também dentro de anexos e apêndices\n\n"

    "Regras obrigatórias:\n"
    "- inclua um item somente se houver título curto, claro e estrutural\n"
    "- não inclua itens apenas porque possuem numeração\n"
    "- não inclua itens quando o texto parecer continuação de parágrafo ou texto corrido\n"
    "- não inclua títulos longos, descritivos ou com aparência de frase completa\n"
    "- não invente título\n"
    "- não complete título truncado\n"
    "- ignore nível 5 ou superior\n"
    "- se houver dúvida, omita\n\n"

    "Regra especial para ANEXOS e APÊNDICES:\n"
    "- depois de identificar um ANEXO ou APÊNDICE, continue examinando cuidadosamente TODAS as páginas pertencentes a ele\n"
    "- procure subtítulos internos curtos, mesmo que não estejam numerados\n"
    "- dentro de ANEXOS e APÊNDICES, também podem existir subitens equivalentes a nível 2, nível 3 e nível 4\n"
    "- esses subitens devem ser incluídos se forem curtos e estruturais\n"
    "- não pare no título principal do anexo/apêndice\n"
    "- só omita subitens do anexo/apêndice quando forem longos, textuais ou duvidosos\n\n"

    "Regra sobre hiatos de páginas:\n"
    "- se qualquer item cobrir muitas páginas sem subtítulos intermediários identificados, examine essas páginas com atenção redobrada procurando cabeçalhos internos curtos\n"
    "- um intervalo grande de páginas sem estrutura detectada é forte indício de que existem subtítulos ainda não capturados\n"
    "- vale tanto para o corpo principal quanto para ANEXOS e APÊNDICES\n"
    "- ainda assim, só inclua se forem curtos, estruturais e funcionarem como rótulo de seção\n\n"

    "Critério para decidir:\n"
    "- o item deve funcionar como rótulo de seção\n"
    "- o item deve ser curto\n"
    "- o item deve parecer visualmente um cabeçalho, e não conteúdo textual\n"
    "- a mesma regra vale para itens numerados e para subitens internos de anexos/apêndices\n\n"

    "Formato da saída:\n"
    "▶ 1 - Título (p. X)\n"
    "    • 1.1 - Subtítulo (p. X)\n"
    "        ◦ 1.1.1 - Subtítulo curto (p. X)\n"
    "            ▪ 1.1.1.1 - Subtítulo curto (p. X)\n"
    "▶ ANEXO I - Título (p. X)\n"
    "    • Subtítulo interno curto do anexo (p. X)\n"
    "        ◦ Subitem curto do anexo (p. X)\n"
    "            ▪ Subitem curto do anexo (p. X)\n"
    "▶ APÊNDICE A - Título (p. X)\n"
    "    • Subtítulo interno curto do apêndice (p. X)\n"
    "        ◦ Subitem curto do apêndice (p. X)\n"
    "            ▪ Subitem curto do apêndice (p. X)\n\n"

    "Retorne somente a lista, sem comentários, sem explicações e sem texto fora do formato."
)

def _carregar_pdf_base64(caminho: str) -> str:
    with open(caminho, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def _contar_paginas(caminho: str) -> int:
    return len(PdfReader(caminho).pages)


def salvar_toc_md(path_pdf: str, toc_md: str) -> Path:
    path_pdf = Path(path_pdf)
    path_saida = path_pdf.with_name(f"{path_pdf.stem}_toc.txt")
    path_saida.write_text(toc_md, encoding="utf-8")
    return path_saida

def nivel_linha(linha: str) -> int:
    linha = linha.lstrip()
    if linha.startswith("▶"):
        return 1
    if linha.startswith("•"):
        return 2
    if linha.startswith("◦"):
        return 3
    if linha.startswith("▪"):
        return 4
    return 99

def extrair_itens(resultado: str):
    itens = []
    padrao = re.compile(
        r"^(?P<raw>\s*[▶•◦▪]\s+.+?)\s+\(p\.\s*(?P<pagina>\d+)(?:–\d+)?\)\s*$"
    )

    for linha in resultado.splitlines():
        linha = linha.rstrip()
        if not linha:
            continue

        m = padrao.match(linha)
        if not m:
            continue

        texto = re.sub(r"^[\s▶•◦▪]+", "", m.group("raw")).strip()

        itens.append({
            "texto": texto,
            "pagina_inicial": int(m.group("pagina")),
            "nivel": nivel_linha(linha),
        })
    return itens


def inferir_janelas(resultado: str, ultima_pagina: int | None = None):
    itens = extrair_itens(resultado)

    for i, item in enumerate(itens):
        fim_aprox = ultima_pagina if ultima_pagina is not None else item["pagina_inicial"]

        for j in range(i + 1, len(itens)):
            prox = itens[j]
            if prox["nivel"] <= item["nivel"]:
                fim_aprox = prox["pagina_inicial"]
                break

        item["pagina_final_aprox"] = fim_aprox

    return itens


def formatar_toc_md(itens):
    linhas = []

    for item in itens:
        if item["nivel"] == 1:
            prefixo = "#"
        elif item["nivel"] == 2:
            prefixo = "##"
        elif item["nivel"] == 3:
            prefixo = "###"
        elif item["nivel"] == 4:
            prefixo = "####"
        else:
            prefixo = "#####"

        linhas.append(
            f'{prefixo} {item["texto"]} (p. {item["pagina_inicial"]}–{item["pagina_final_aprox"]})'
        )

    return "\n".join(linhas)

def get_toc_md(path_pdf: str, provider: str = "anthropic", modelo: str = "sonnet", max_tokens: int = MAX_TOKENS_TOC) -> str:
    """Extrai, processa e retorna TOC formatado do PDF."""
    pdf_b64 = _carregar_pdf_base64(path_pdf)
    
    if provider == "anthropic":
        llm = ChatAnthropic(model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0)
        bloco_pdf = {
            "type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
        }
    else:
        llm = ChatGoogleGenerativeAI(model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0)
        bloco_pdf = {"type": "media", "mime_type": "application/pdf", "data": pdf_b64}

    mensagem = HumanMessage(content=[bloco_pdf, {"type": "text", "text": PROMPT}])
    resposta = llm.invoke([mensagem])
    
    conteudo = resposta.content
    if isinstance(conteudo, list):
        conteudo = "".join(b["text"] for b in conteudo if isinstance(b, dict) and b.get("type") == "text")
    
    ultima_pagina = _contar_paginas(path_pdf)
    itens = inferir_janelas(conteudo, ultima_pagina=ultima_pagina)
    return formatar_toc_md(itens)

In [5]:
path_pdf = '/home/julio/Documentos/tcc_GENAI/demo0/edital-assistant/data/editais/edital-1-2024-abertura-concurso-cvm.pdf'

# Usar
toc_md = get_toc_md(path_pdf, provider="anthropic", modelo="sonnet")
print(toc_md)

# Salvar
path_saida = salvar_toc_md(path_pdf, toc_md)
print(f"Salvo em: {path_saida}")

# 1 - Das Disposições Preliminares (p. 1–1)
# 2 - Do Concurso (p. 1–1)
# 3 - Dos Cargos e Especialidades (p. 1–3)
# 4 - Das Inscrições (p. 3–4)
# 5 - Da Isenção da Taxa de Inscrição (p. 4–5)
# 6 - Das Vagas Destinadas a Pessoas com Deficiência (p. 5–7)
# 7 - Do Atendimento a Candidatos com Necessidades de Adaptações para Realização das Provas (p. 7–8)
# 8 - Das Vagas Destinadas a Candidatos Negros (p. 8–10)
# 9 - Das Provas (p. 10–13)
## 9.6 - Da Prova Objetiva (p. 10–12)
## 9.7 - Da Prova Discursiva (p. 12–13)
# 10 - Da Realização das Provas (p. 13–16)
# 11 - Da Classificação no Concurso (p. 16–16)
# 12 - Dos Critérios de Desempate (p. 16–16)
# 13 - Dos Recursos (p. 16–17)
# 14 - Da Homologação e Nomeação (p. 17–18)
# 15 - Das Disposições Finais (p. 18–19)
# ANEXO I - Conteúdo Programático (p. 19–34)
## Conhecimentos Básicos (p. 19–20)
### 1 - Língua Portuguesa (p. 19–19)
### 2 - Estrutura do Mercado de Valores Mobiliários (MVM) (p. 19–19)
### 3 - Fundamentos de Direito (p. 19–20)
###

In [6]:
path_pdf = '/home/julio/Documentos/tcc_GENAI/demo0/edital-assistant/data/editais/20240812EditalRetificadoFINALBNDES12agostoBNDES.pdf'

# Usar
toc_md = get_toc_md(path_pdf, provider="anthropic", modelo="sonnet")
print(toc_md)

# Salvar
path_saida = salvar_toc_md(path_pdf, toc_md)
print(f"Salvo em: {path_saida}")

# 1 - Das Disposições Preliminares (p. 1–2)
# 2 - Do Cargo, Atribuições do Cargo, Remuneração Inicial, Jornada de Trabalho, Ênfases e Requisitos Básicos (p. 2–4)
# 3 - Das Vagas Reservadas (p. 4–8)
## 3.1 - Das Vagas Reservadas às Pessoas com Deficiência (PcD) (p. 4–6)
## 3.2 - Das Vagas Reservadas às Pessoas Negras (p. 6–6)
## 3.3 - Do Procedimento de Heteroidentificação das Pessoas Negras (p. 6–8)
# 4 - Dos Requisitos e das Condições para Investidura no Cargo/Ênfase (p. 8–9)
# 5 - Da Classificação e Convocação (p. 9–9)
## 5.4 - Da Ordem de Convocação por Cargo/Ênfase (p. 9–9)
# 6 - Das Inscrições na Seleção Pública (p. 9–13)
## 6.3 - Inscrições (p. 10–12)
## 6.15 - Da Solicitação de Adaptações Razoáveis (p. 12–13)
# 7 - Da Confirmação de Inscrição (p. 13–13)
# 8 - Da Etapa de Qualificação Técnica (p. 13–15)
# 9 - Das Normas e Procedimentos Relativos à Realização das Provas (p. 15–18)
# 10 - Dos Recursos e da Revisão (p. 18–18)
# 11 - Da Admissão (p. 18–19)
# 12 - Das Disposições Fina

In [7]:
path_pdf = '/home/julio/Documentos/tcc_GENAI/demo0/edital-assistant/data/editais/editalpetroleobrasileiro.pdf'

# Usar
toc_md = get_toc_md(path_pdf, provider="anthropic", modelo="sonnet")
print(toc_md)

# Salvar
path_saida = salvar_toc_md(path_pdf, toc_md)
print(f"Salvo em: {path_saida}")

# 1 - Das Disposições Preliminares (p. 1–1)
# 2 - Das Ênfases (p. 1–1)
# 3 - Das Reservas de Vagas (p. 1–6)
## 3.1 - Das Vagas Destinadas às Pessoas com Deficiência (PCD) (p. 1–4)
### 3.1.12 - Da avaliação da equipe multiprofissional (p. 3–4)
## 3.2 - Das vagas destinadas aos(as) candidatos(as) negros(as) (p. 4–6)
### 3.2.2 - Do procedimento de heteroidentificação complementar à autodeclaração dos(as) candidatos(as) negros(as) (p. 4–6)
# 4 - Dos Requisitos Básicos Exigidos para Admissão ou Readmissão (p. 6–6)
# 5 - Das Inscrições no Processo Seletivo Público (p. 6–11)
## 5.4 - Das disposições gerais sobre a inscrição no processo seletivo público (p. 6–11)
### 5.4.8 - Dos procedimentos para o pedido de isenção de taxa de inscrição (p. 8–9)
### 5.4.9 - Dos procedimentos para a solicitação de atendimento especial (p. 9–11)
# 6 - Das Fases do Processo Seletivo Público (p. 11–12)
# 7 - Das Provas Objetivas (p. 12–13)
## 7.11 - Dos Critérios de Avaliação das Provas Objetivas (p. 12–13)
## 7.

# Split arquives

In [8]:
import re
from pathlib import Path
import pandas as pd
from pypdf import PdfReader, PdfWriter

EDITAIS_DIR = Path('data/editais')
SPLITS_DIR  = EDITAIS_DIR / 'splits'

# Só cabeçalhos de nível 1:  # <titulo> (p. <ini>–<fim>)
# '^#\s' garante um único '#' (descarta '##', '###', ...). Aceita '–' e '-'.
LEVEL1_RE = re.compile(
    r'^#\s+(?P<title>.+?)\s+\(p\.\s*(?P<start>\d+)\s*[–\-]\s*(?P<end>\d+)\)\s*$'
)

def parse_toc_level1(toc_path: Path) -> list[dict]:
    chapters = []
    for line in toc_path.read_text(encoding='utf-8').splitlines():
        m = LEVEL1_RE.match(line)
        if m:
            chapters.append({
                'title':      m.group('title').strip(),
                'start_page': int(m.group('start')),
                'end_page':   int(m.group('end')),
            })
    return chapters

def write_pages(reader: PdfReader, start_1b: int, end_1b: int, out_path: Path) -> None:
    """Grava um PDF com as páginas [start_1b..end_1b] (1-based, ambas inclusivas)."""
    writer = PdfWriter()
    start_idx = max(0, start_1b - 1)
    end_idx   = min(len(reader.pages), end_1b)   # end_1b inclusivo ↔ range exclusivo
    for i in range(start_idx, end_idx):
        writer.add_page(reader.pages[i])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'wb') as f:
        writer.write(f)

def split_edital(pdf_path: Path, toc_path: Path, out_dir: Path) -> list[dict]:
    chapters = parse_toc_level1(toc_path)
    reader = PdfReader(str(pdf_path))
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    # cap_0: páginas 1 até start_page do primeiro capítulo (inclusivas)
    first_start = chapters[0]['start_page']
    cap0_path = out_dir / 'cap_0.pdf'
    write_pages(reader, 1, first_start, cap0_path)
    rows.append({
        'chapter_num': 0,
        'title':       '(conteúdo antes do primeiro capítulo)',
        'start_page':  1,
        'end_page':    first_start,
        'cap_path':    str(cap0_path),
    })

    # cap_1, cap_2, ...
    for idx, chap in enumerate(chapters, start=1):
        out_path = out_dir / f'cap_{idx}.pdf'
        write_pages(reader, chap['start_page'], chap['end_page'], out_path)
        rows.append({
            'chapter_num': idx,
            'title':       chap['title'],
            'start_page':  chap['start_page'],
            'end_page':    chap['end_page'],
            'cap_path':    str(out_path),
        })
    return rows

# --- Execução ----------------------------------------------------------------
all_rows = []
for toc_path in sorted(EDITAIS_DIR.glob('*_toc.txt')):
    edital_name = toc_path.stem.removesuffix('_toc')
    pdf_path = EDITAIS_DIR / f'{edital_name}.pdf'
    if not pdf_path.exists():
        print(f'[skip] PDF não encontrado: {pdf_path}')
        continue
    out_dir = SPLITS_DIR / edital_name
    rows = split_edital(pdf_path, toc_path, out_dir)
    for r in rows:
        r['edital_filename'] = edital_name
        r['edital_pdf_path'] = str(pdf_path)
    all_rows.extend(rows)
    print(f'[ok]   {edital_name}: {len(rows)} capítulos → {out_dir}')

df = pd.DataFrame(all_rows, columns=[
    'edital_filename', 'chapter_num', 'title',
    'start_page', 'end_page', 'cap_path', 'edital_pdf_path',
])
df

[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES: 17 capítulos → data/editais/splits/20240812EditalRetificadoFINALBNDES12agostoBNDES
[ok]   edital-1-2024-abertura-concurso-cvm: 23 capítulos → data/editais/splits/edital-1-2024-abertura-concurso-cvm
[ok]   editalpetroleobrasileiro: 23 capítulos → data/editais/splits/editalpetroleobrasileiro


,edital_filename,chapter_num,title,start_page,end_page,cap_path,edital_pdf_path
0,20240812EditalRetificadoFINALBNDES12agostoBNDES,0,(conteúdo antes do primeiro capítulo),1,1,data/editais/splits/20240812EditalRetificadoFI...,data/editais/20240812EditalRetificadoFINALBNDE...
1,20240812EditalRetificadoFINALBNDES12agostoBNDES,1,1 - Das Disposições Preliminares,1,2,data/editais/splits/20240812EditalRetificadoFI...,data/editais/20240812EditalRetificadoFINALBNDE...
2,20240812EditalRetificadoFINALBNDES12agostoBNDES,2,"2 - Do Cargo, Atribuições do Cargo, Remuneraçã...",2,4,data/editais/splits/20240812EditalRetificadoFI...,data/editais/20240812EditalRetificadoFINALBNDE...
3,20240812EditalRetificadoFINALBNDES12agostoBNDES,3,3 - Das Vagas Reservadas,4,8,data/editais/splits/20240812EditalRetificadoFI...,data/editais/20240812EditalRetificadoFINALBNDE...
4,20240812EditalRetificadoFINALBNDES12agostoBNDES,4,4 - Dos Requisitos e das Condições para Invest...,8,9,data/editais/splits/20240812EditalRetificadoFI...,data/editais/20240812EditalRetificadoFINALBNDE...
...,...,...,...,...,...,...,...
58,editalpetroleobrasileiro,18,"ANEXO II - Cargo, Ênfases, Requisitos, Síntese...",25,31,data/editais/splits/editalpetroleobrasileiro/c...,data/editais/editalpetroleobrasileiro.pdf
59,editalpetroleobrasileiro,19,ANEXO III - Objetos de Avaliação,31,47,data/editais/splits/editalpetroleobrasileiro/c...,data/editais/editalpetroleobrasileiro.pdf
60,editalpetroleobrasileiro,20,ANEXO IV - Cronograma,47,49,data/editais/splits/editalpetroleobrasileiro/c...,data/editais/editalpetroleobrasileiro.pdf
61,editalpetroleobrasileiro,21,ANEXO V - Modelo de Laudo Médico,49,50,data/editais/splits/editalpetroleobrasileiro/c...,data/editais/editalpetroleobrasileiro.pdf


In [ ]:
import base64
from pathlib import Path

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

MODELOS = {
    "anthropic": {
        "sonnet": "claude-sonnet-4-5",
        "haiku":  "claude-haiku-4-5",
    },
    "google": {
        "flash":      "gemini-2.5-flash",
        "flash-lite": "gemini-3.1-flash-lite-preview",
    },
}

MAX_TOKENS_CAP = 32000

PROMPT_EXTRACAO_CAP = """\
Você vai receber em PDF UM capítulo de um edital de concurso público brasileiro.
Sua tarefa é extrair o conteúdo como markdown, aplicando o filtro descrito abaixo.

CONTEXTO DO LEITOR:
- O leitor é um candidato que vai prestar este concurso APENAS na ênfase "Ciência de Dados".
- Ele NÃO vai se inscrever em nenhuma outra ênfase.
- Conteúdo específico de OUTRAS ênfases é irrelevante e deve ser OMITIDO.
- Conteúdo comum (aplicável a qualquer candidato, independente da ênfase) é essencial e deve ser PRESERVADO.

REGRAS DE EXTRAÇÃO (sobre o texto que você mantiver):
1. Preserve o texto ORIGINAL, palavra por palavra. NÃO resuma, NÃO parafraseie, NÃO simplifique.
2. Mantenha a estrutura hierárquica (títulos, subtítulos, numeração 1., 1.1, 1.1.1, alíneas a), b), etc.).
3. Formate como markdown limpo: `#` para o título do capítulo, `##`/`###` para subseções, listas com `-` ou `1.`, negrito/itálico quando presentes no original.
4. Preserve a numeração exata dos itens (1.1, 1.2.3, I, II, a), b), etc.), exatamente como aparece.
5. Se houver tabelas, reproduza como tabelas markdown.

REGRAS DE FILTRAGEM:
MANTENHA integralmente:
- Toda seção cujo conteúdo se aplique a qualquer candidato (ex.: "Conhecimentos Básicos", regras gerais, disposições comuns a todos os cargos/ênfases, Língua Portuguesa, Raciocínio Lógico, etc.).
- Toda seção específica da ênfase "Ciência de Dados" — inclua-a na ÍNTEGRA, sem cortar.

OMITA:
- Seções cujo conteúdo seja EXCLUSIVO de OUTRAS ênfases/cargos/áreas (ex.: Mecânica, Química, Direito, Administração, Engenharia Civil, Geofísica, etc., quando NÃO forem Ciência de Dados).
- Omita LIMPAMENTE: não deixe placeholder, não escreva "[seção omitida]", não comente o que foi removido, não liste as ênfases omitidas. Simplesmente pule para a próxima seção relevante.

EM CASO DE DÚVIDA, PRESERVE. É preferível incluir algo ambíguo a perder conteúdo relevante ao candidato.

SAÍDA:
- Retorne APENAS o markdown extraído e filtrado.
- Não adicione comentários, preâmbulos ou observações suas.
- Não envolva a saída em ```markdown ... ```.
- Comece direto pelo primeiro título ou linha relevante do capítulo.
"""


def _carregar_pdf_base64(path_pdf: str | Path) -> str:
    return base64.b64encode(Path(path_pdf).read_bytes()).decode("utf-8")


def extrair_md_capitulo(
    path_pdf_cap: str | Path,
    provider: str = "google",
    modelo: str = "flash-lite",
    max_tokens: int = MAX_TOKENS_CAP,
    prompt: str = PROMPT_EXTRACAO_CAP,
) -> str:
    pdf_b64 = _carregar_pdf_base64(path_pdf_cap)

    if provider == "anthropic":
        llm = ChatAnthropic(
            model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0
        )
        bloco_pdf = {
            "type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
        }
    else:
        llm = ChatGoogleGenerativeAI(
            model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0
        )
        bloco_pdf = {"type": "media", "mime_type": "application/pdf", "data": pdf_b64}

    msg = HumanMessage(content=[bloco_pdf, {"type": "text", "text": prompt}])
    resp = llm.invoke([msg])
    return resp.content if isinstance(resp.content, str) else resp.content[0]["text"]


# --- Teste: ANEXO III (cap_19) do Petrobras ---------------------------------
cap_path = Path("data/editais/splits/editalpetroleobrasileiro/cap_19.pdf")
md = extrair_md_capitulo(cap_path, provider="google", modelo="flash-lite")

out_path = cap_path.with_suffix(".md")
out_path.write_text(md, encoding="utf-8")

print(f"Salvo em: {out_path}")
print(f"Tamanho: {len(md):,} chars | {md.count(chr(10)):,} linhas")
print("--- prévia (primeiros 3000 chars) ---")
print(md[:3000])

Salvo em: data/editais/splits/editalpetroleobrasileiro/cap_19.md
Tamanho: 8,798 chars | 78 linhas
--- prévia (primeiros 3000 chars) ---
PETRÓLEO BRASILEIRO S.A. - PETROBRAS
PSP RH 2021.1

ANEXO III – OBJETOS DE AVALIAÇÃO 

CONHECIMENTOS BÁSICOS 

LÍNGUA PORTUGUESA 1 Compreensão e interpretação de textos de gêneros variados. 2 Reconhecimento de 33 tipos e 
gêneros textuais. 3 Domínio da ortografia oficial. 4 Domínio dos mecanismos de coesão textual. 4.1 Emprego de elementos de 
referenciação, substituição e repetição, de conectores e de outros elementos de sequenciação textual. 4.2 Emprego de tempos 
e modos verbais. 5 Domínio da estrutura morfossintática do período. 5.1 Emprego das classes de palavras. 5.2 Relações de 
coordenação entre orações e entre termos da oração. 5.3 Relações de subordinação entre orações e entre termos da oração. 
5.4 Emprego dos sinais de pontuação. 5.5 Concordância verbal e nominal. 5.6 Regência verbal e nominal. 5.7 Emprego do sinal 
indicativo de crase. 5.8

In [13]:
MODELOS = {
    "anthropic": {
        "sonnet": "claude-sonnet-4-5",
        "haiku":  "claude-haiku-4-5",
    },
    "google": {
        "flash":      "gemini-2.5-flash",
        "flash-lite": "gemini-3.1-flash-lite-preview",
    },
}

PROMPT_EXTRACAO_CAP = """\
Você vai receber em PDF um trecho de um edital de concurso público brasileiro.
Esse trecho contém o capítulo "{titulo_capitulo}" e pode conter, na última página, o início do capítulo seguinte{trecho_proximo}.

Sua tarefa: extrair APENAS o conteúdo do capítulo "{titulo_capitulo}", formatado como markdown, aplicando o filtro descrito abaixo.

LIMITES DO CAPÍTULO:
- COMECE no título "{titulo_capitulo}" (ou na primeira linha relevante dele).
- PARE imediatamente ao encontrar o início do próximo capítulo{trecho_parada}. NÃO inclua o título do próximo capítulo nem qualquer linha dele.
- Ignore cabeçalhos/rodapés de página repetidos (nome do edital, paginação, etc.).

CONTEXTO DO LEITOR:
- O leitor é um candidato que vai prestar este concurso APENAS na ênfase "Ciência de Dados".
- Ele NÃO vai se inscrever em nenhuma outra ênfase.
- Conteúdo específico de OUTRAS ênfases é irrelevante e deve ser OMITIDO.
- Conteúdo comum (aplicável a qualquer candidato, independente da ênfase) é essencial e deve ser PRESERVADO.

REGRAS DE EXTRAÇÃO (sobre o texto que você mantiver):
1. Preserve o texto ORIGINAL, palavra por palavra. NÃO resuma, NÃO parafraseie, NÃO simplifique.
2. Mantenha a estrutura hierárquica (títulos, subtítulos, numeração 1., 1.1, 1.1.1, alíneas a), b), etc.).
3. Formate como markdown limpo: `#` para o título do capítulo, `##`/`###` para subseções, listas com `-` ou `1.`, negrito/itálico quando presentes no original.
4. Preserve a numeração exata dos itens (1.1, 1.2.3, I, II, a), b), etc.), exatamente como aparece.
5. Se houver tabelas, reproduza como tabelas markdown.

REGRAS DE FILTRAGEM:
MANTENHA integralmente:
- Toda seção cujo conteúdo se aplique a qualquer candidato (ex.: "Conhecimentos Básicos", regras gerais, Língua Portuguesa, Raciocínio Lógico, etc.).
- Toda seção específica da ênfase "Ciência de Dados" — inclua-a na ÍNTEGRA, sem cortar.

OMITA:
- Seções cujo conteúdo seja EXCLUSIVO de OUTRAS ênfases/cargos/áreas.
- Omita LIMPAMENTE: não deixe placeholder, não escreva "[seção omitida]", não comente o que foi removido, não liste as ênfases omitidas.

EM CASO DE DÚVIDA, PRESERVE.

SAÍDA:
- Retorne APENAS o markdown extraído e filtrado.
- Não adicione comentários, preâmbulos ou observações.
- Não envolva a saída em ```markdown ... ```.
- Comece direto pelo título do capítulo.
"""


def extrair_md_capitulo(
    path_pdf_cap: str | Path,
    titulo_capitulo: str,
    titulo_proximo: str | None = None,
    provider: str = "anthropic",
    modelo: str = "haiku",
    max_tokens: int = MAX_TOKENS_CAP,
) -> str:
    pdf_b64 = _carregar_pdf_base64(path_pdf_cap)

    if titulo_proximo:
        trecho_proximo = f' ("{titulo_proximo}")'
        trecho_parada  = f' — identificado pelo título "{titulo_proximo}"'
    else:
        trecho_proximo = ""
        trecho_parada  = ""

    prompt = PROMPT_EXTRACAO_CAP.format(
        titulo_capitulo=titulo_capitulo,
        trecho_proximo=trecho_proximo,
        trecho_parada=trecho_parada,
    )

    if provider == "anthropic":
        llm = ChatAnthropic(
            model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0
        )
        bloco_pdf = {
            "type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
        }
    else:
        llm = ChatGoogleGenerativeAI(
            model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0
        )
        bloco_pdf = {"type": "media", "mime_type": "application/pdf", "data": pdf_b64}

    msg = HumanMessage(content=[bloco_pdf, {"type": "text", "text": prompt}])
    resp = llm.invoke([msg])
    return resp.content if isinstance(resp.content, str) else resp.content[0]["text"]


# --- Teste: cap_19 do Petrobras usando os títulos já existentes no df -------
edital = "editalpetroleobrasileiro"
cap_num = 19

linha_atual   = df[(df.edital_filename == edital) & (df.chapter_num == cap_num)].iloc[0]
prox = df[(df.edital_filename == edital) & (df.chapter_num == cap_num + 1)]
titulo_proximo = prox.iloc[0]["title"] if len(prox) else None

md = extrair_md_capitulo(
    path_pdf_cap   = linha_atual["cap_path"],
    titulo_capitulo= linha_atual["title"],
    titulo_proximo = titulo_proximo,
    provider       = "anthropic",
    modelo         = "haiku",
)

out_path = Path(linha_atual["cap_path"]).with_suffix(".md")
out_path.write_text(md, encoding="utf-8")

print(f"Capítulo: {linha_atual['title']}")
print(f"Para em: {titulo_proximo}")
print(f"Salvo em: {out_path}")
print(f"Tamanho: {len(md):,} chars")
print("--- prévia ---")
print(md[:2000])

Capítulo: ANEXO III - Objetos de Avaliação
Para em: ANEXO IV - Cronograma
Salvo em: data/editais/splits/editalpetroleobrasileiro/cap_19.md
Tamanho: 6,230 chars
--- prévia ---
# ANEXO III – OBJETOS DE AVALIAÇÃO

## CONHECIMENTOS BÁSICOS

### LÍNGUA PORTUGUESA

1. Compreensão e interpretação de textos de gêneros variados.
2. Reconhecimento de 33 tipos e gêneros textuais.
3. Domínio da ortografia oficial.
4. Domínio dos mecanismos de coesão textual.
   - 4.1 Emprego de elementos de referenciação, substituição e repetição, de conectores e de outros elementos de sequenciação textual.
   - 4.2 Emprego de tempos e modos verbais.
5. Domínio da estrutura morfossintática do período.
   - 5.1 Emprego das classes de palavras.
   - 5.2 Relações de coordenação entre orações e entre termos da oração.
   - 5.3 Relações de subordinação entre orações e entre termos da oração.
   - 5.4 Emprego dos sinais de pontuação.
   - 5.5 Concordância verbal e nominal.
   - 5.6 Regência verbal e nominal.
   - 5.7 Em

In [14]:
PULAR = [0, 19]   # caps a pular

edital = "editalpetroleobrasileiro"
df_edital = df[df.edital_filename == edital].sort_values("chapter_num").reset_index(drop=True)

for _, linha in df_edital.iterrows():
    cap_num = int(linha["chapter_num"])

    if cap_num in PULAR:
        print(f"[skip] cap_{cap_num}: {linha['title']}")
        continue
    if linha["cap_path"] is None or (isinstance(linha["cap_path"], float)):
        print(f"[skip] cap_{cap_num}: sem arquivo")
        continue

    prox = df_edital[df_edital.chapter_num == cap_num + 1]
    titulo_proximo = prox.iloc[0]["title"] if len(prox) else None

    md = extrair_md_capitulo(
        path_pdf_cap   = linha["cap_path"],
        titulo_capitulo= linha["title"],
        titulo_proximo = titulo_proximo,
        provider       = "anthropic",
        modelo         = "haiku",
    )

    out_path = Path(linha["cap_path"]).with_suffix(".md")
    out_path.write_text(md, encoding="utf-8")
    print(f"[ok]   cap_{cap_num}: {len(md):>6,} chars → {out_path.name}")

[skip] cap_0: (conteúdo antes do primeiro capítulo)
[ok]   cap_1:  1,913 chars → cap_1.md
[ok]   cap_2:    173 chars → cap_2.md
[ok]   cap_3: 20,313 chars → cap_3.md
[ok]   cap_4:  1,497 chars → cap_4.md
[ok]   cap_5: 22,249 chars → cap_5.md
[ok]   cap_6:  2,114 chars → cap_6.md
[ok]   cap_7:  7,412 chars → cap_7.md
[ok]   cap_8: 11,054 chars → cap_8.md
[ok]   cap_9:    325 chars → cap_9.md
[ok]   cap_10:  4,354 chars → cap_10.md
[ok]   cap_11:  1,848 chars → cap_11.md
[ok]   cap_12:  3,837 chars → cap_12.md
[ok]   cap_13:  3,272 chars → cap_13.md
[ok]   cap_14:  4,155 chars → cap_14.md
[ok]   cap_15:    242 chars → cap_15.md
[ok]   cap_16:  7,142 chars → cap_16.md
[ok]   cap_17:    583 chars → cap_17.md
[ok]   cap_18:    865 chars → cap_18.md
[skip] cap_19: ANEXO III - Objetos de Avaliação
[ok]   cap_20:  4,827 chars → cap_20.md
[ok]   cap_21:  1,151 chars → cap_21.md
[ok]   cap_22:  2,135 chars → cap_22.md


In [15]:
PROMPT_CAP0 = """\
Você vai receber em PDF a primeira página (ou páginas iniciais) de um edital de concurso público brasileiro.
Essa parte vem ANTES do primeiro capítulo numerado e normalmente contém: órgão/empresa responsável, título do edital, identificação do processo seletivo, data, e eventualmente um preâmbulo.

Sua tarefa: extrair o conteúdo como TEXTO CORRIDO ORGANIZADO.

LIMITES:
- PARE imediatamente ao encontrar o início do primeiro capítulo numerado{trecho_parada}. NÃO inclua o título nem qualquer linha dele.
- Ignore cabeçalhos/rodapés de página repetidos e paginação.

REGRAS:
1. Preserve o texto ORIGINAL, palavra por palavra. NÃO resuma, NÃO parafraseie.
2. NÃO use formatação markdown: nada de `#`, `##`, `**negrito**`, listas com `-` ou `1.`, etc.
3. Organize em parágrafos simples, separados por linha em branco, na ordem em que aparecem no documento.
4. Mantenha linha própria apenas para elementos que naturalmente ficam sozinhos (ex.: nome do órgão, título do edital, data) — sem marcadores.
5. Se houver numeração original no texto (ex.: "1.1", "I -"), preserve-a como parte do texto, mas sem transformar em lista markdown.

SAÍDA:
- Retorne APENAS o texto extraído.
- Não adicione comentários, preâmbulos ou observações.
- Não envolva em ```...```.
"""


def extrair_texto_cap0(
    path_pdf_cap: str | Path,
    titulo_proximo: str | None = None,
    provider: str = "anthropic",
    modelo: str = "haiku",
    max_tokens: int = MAX_TOKENS_CAP,
) -> str:
    pdf_b64 = _carregar_pdf_base64(path_pdf_cap)
    trecho_parada = f' ("{titulo_proximo}")' if titulo_proximo else ""
    prompt = PROMPT_CAP0.format(trecho_parada=trecho_parada)

    if provider == "anthropic":
        llm = ChatAnthropic(model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0)
        bloco_pdf = {
            "type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
        }
    else:
        llm = ChatGoogleGenerativeAI(model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0)
        bloco_pdf = {"type": "media", "mime_type": "application/pdf", "data": pdf_b64}

    msg = HumanMessage(content=[bloco_pdf, {"type": "text", "text": prompt}])
    resp = llm.invoke([msg])
    return resp.content if isinstance(resp.content, str) else resp.content[0]["text"]


# --- Rodar só o cap_0 do Petrobras ------------------------------------------
edital = "editalpetroleobrasileiro"
df_edital = df[df.edital_filename == edital].sort_values("chapter_num").reset_index(drop=True)

linha_cap0 = df_edital[df_edital.chapter_num == 0].iloc[0]
linha_cap1 = df_edital[df_edital.chapter_num == 1].iloc[0]

texto = extrair_texto_cap0(
    path_pdf_cap   = linha_cap0["cap_path"],
    titulo_proximo = linha_cap1["title"],
    provider       = "anthropic",
    modelo         = "haiku",
)

out_path = Path(linha_cap0["cap_path"]).with_suffix(".md")
out_path.write_text(texto, encoding="utf-8")
print(f"Salvo em: {out_path}")
print(f"Tamanho: {len(texto):,} chars")
print("--- prévia ---")
print(texto[:1500])

Salvo em: data/editais/splits/editalpetroleobrasileiro/cap_0.md
Tamanho: 377 chars
--- prévia ---
PETRÓLEO BRASILEIRO S.A. – PETROBRAS

PROCESSO SELETIVO PÚBLICO PARA PREENCHIMENTO DE VAGAS E FORMAÇÃO DE CADASTRO EM CARGOS DE NÍVEL SUPERIOR

EDITAL N.º 1 – PETROBRAS/PSP RH 2021, DE 15 DE DEZEMBRO DE 2021

PETRÓLEO BRASILEIRO S.A. (PETROBRAS) realizará processo seletivo público para provimento de vagas e formação de cadastro, mediante condições estabelecidas neste edital.


In [18]:
from pathlib import Path

def processar_edital(
    edital: str,
    df,
    pular: list[int] | None = None,
    provider: str = "anthropic",
    modelo: str = "haiku",
) -> None:
    """Extrai MD de todos os capítulos de um edital. cap_0 vira texto corrido; demais, markdown filtrado."""
    pular = set(pular or [])
    df_edital = df[df.edital_filename == edital].sort_values("chapter_num").reset_index(drop=True)

    for _, linha in df_edital.iterrows():
        cap_num  = int(linha["chapter_num"])
        cap_path = linha["cap_path"]

        if cap_num in pular or not isinstance(cap_path, str):
            print(f"[skip] {edital} cap_{cap_num}")
            continue

        prox = df_edital[df_edital.chapter_num == cap_num + 1]
        titulo_proximo = prox.iloc[0]["title"] if len(prox) else None

        if cap_num == 0:
            md = extrair_texto_cap0(cap_path, titulo_proximo, provider, modelo)
        else:
            md = extrair_md_capitulo(cap_path, linha["title"], titulo_proximo, provider, modelo)

        out_path = Path(cap_path).with_suffix(".md")
        out_path.write_text(md, encoding="utf-8")
        print(f"[ok]   {edital} cap_{cap_num}: {len(md):>6,} chars → {out_path.name}")


# --- Rodar todos os editais -------------------------------------------------
edital = 'edital-1-2024-abertura-concurso-cvm'
processar_edital(edital, df, provider="anthropic", modelo="haiku")

[ok]   edital-1-2024-abertura-concurso-cvm cap_0:    751 chars → cap_0.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_1:  1,424 chars → cap_1.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_2:  1,126 chars → cap_2.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_3:  3,709 chars → cap_3.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_4:  8,020 chars → cap_4.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_5:  4,290 chars → cap_5.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_6: 10,398 chars → cap_6.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_7:  6,639 chars → cap_7.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_8:  6,338 chars → cap_8.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_9:  8,984 chars → cap_9.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_10: 11,945 chars → cap_10.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_11:  1,153 chars → cap_11.md
[ok]   edital-1-2024-abertura-concurso-cvm cap_12:  1,131 chars → cap_12.md
[ok]   edital-1-2024-abertura-co

In [19]:
from pathlib import Path
import re 

RE_NUMERO_HIER = re.compile(r"^(\d+\.\d+(?:\.\d+)*)\.?\s+(.*)$")

def aplicar_hierarquia_md(txt: str) -> str:
    linhas_out = []
    for linha in txt.splitlines():
        m = RE_NUMERO_HIER.match(linha.strip())
        if not m:
            linhas_out.append(linha)          # não mexe
            continue
        numero, resto = m.group(1), m.group(2)
        nivel = min(numero.count(".") + 1, 6) # 2.1→2, 2.1.1→3, ...
        linhas_out.append(f"{'#' * nivel} {numero}. {resto}")
    return "\n".join(linhas_out)

def atualizar_mds(edital: str, df) -> None:
    """Aplica aplicar_hierarquia_md em todos os .md do edital (exceto cap_0)."""
    df_edital = df[df.edital_filename == edital]
    for _, linha in df_edital.iterrows():
        cap_num  = int(linha["chapter_num"])
        cap_path = linha["cap_path"]
        if cap_num == 0 or not isinstance(cap_path, str):
            continue
        md_path = Path(cap_path).with_suffix(".md")
        if not md_path.exists():
            continue
        original = md_path.read_text(encoding="utf-8")
        md_path.write_text(aplicar_hierarquia_md(original), encoding="utf-8")
        print(f"[ok] {md_path.name}")


atualizar_mds("edital-1-2024-abertura-concurso-cvm", df)

[ok] cap_1.md
[ok] cap_2.md
[ok] cap_3.md
[ok] cap_4.md
[ok] cap_5.md
[ok] cap_6.md
[ok] cap_7.md
[ok] cap_8.md
[ok] cap_9.md
[ok] cap_10.md
[ok] cap_11.md
[ok] cap_12.md
[ok] cap_13.md
[ok] cap_14.md
[ok] cap_15.md
[ok] cap_16.md
[ok] cap_17.md
[ok] cap_18.md
[ok] cap_19.md
[ok] cap_20.md
[ok] cap_21.md
[ok] cap_22.md


In [21]:
edital = '20240812EditalRetificadoFINALBNDES12agostoBNDES'
processar_edital(edital, df, provider="anthropic", modelo="haiku")

[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_0:    993 chars → cap_0.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_1:  3,437 chars → cap_1.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_2:  1,751 chars → cap_2.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_3: 21,065 chars → cap_3.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_4:  2,506 chars → cap_4.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_5:  2,362 chars → cap_5.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_6: 18,133 chars → cap_6.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_7:  1,917 chars → cap_7.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_8:  8,644 chars → cap_8.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_9: 11,569 chars → cap_9.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBNDES cap_10:  2,525 chars → cap_10.md
[ok]   20240812EditalRetificadoFINALBNDES12agostoBND

In [20]:
df['edital_filename'].unique()

<ArrowStringArray>
['20240812EditalRetificadoFINALBNDES12agostoBNDES',
             'edital-1-2024-abertura-concurso-cvm',
                        'editalpetroleobrasileiro']
Length: 3, dtype: str

In [22]:
from pathlib import Path

def atualizar_mds(edital: str, df) -> None:
    """Aplica aplicar_hierarquia_md nos .md do edital, pulando cap_0, anexos e apêndices."""
    df_edital = df[df.edital_filename == edital]
    for _, linha in df_edital.iterrows():
        cap_num  = int(linha["chapter_num"])
        cap_path = linha["cap_path"]
        titulo   = (linha["title"] or "").upper()

        if cap_num == 0 or not isinstance(cap_path, str):
            continue
        if titulo.startswith(("ANEXO", "APÊNDICE", "APENDICE")):
            print(f"[skip anexo/apêndice] cap_{cap_num}: {linha['title']}")
            continue

        md_path = Path(cap_path).with_suffix(".md")
        if not md_path.exists():
            continue
        original = md_path.read_text(encoding="utf-8")
        md_path.write_text(aplicar_hierarquia_md(original), encoding="utf-8")
        print(f"[ok] {md_path.name}")


for edital in ['20240812EditalRetificadoFINALBNDES12agostoBNDES',
                        'editalpetroleobrasileiro']:
    atualizar_mds(edital, df)

[ok] cap_1.md
[ok] cap_2.md
[ok] cap_3.md
[ok] cap_4.md
[ok] cap_5.md
[ok] cap_6.md
[ok] cap_7.md
[ok] cap_8.md
[ok] cap_9.md
[ok] cap_10.md
[ok] cap_11.md
[ok] cap_12.md
[skip anexo/apêndice] cap_13: ANEXO I - Cargo, Ênfases, Vagas e Cadastro de Reserva
[skip anexo/apêndice] cap_14: ANEXO II - Conteúdos Programáticos
[skip anexo/apêndice] cap_15: ANEXO III - Cronograma
[skip anexo/apêndice] cap_16: ANEXO IV - Modelo de Relatório/Laudo Caracterizador de Deficiência
[ok] cap_1.md
[ok] cap_2.md
[ok] cap_3.md
[ok] cap_4.md
[ok] cap_5.md
[ok] cap_6.md
[ok] cap_7.md
[ok] cap_8.md
[ok] cap_9.md
[ok] cap_10.md
[ok] cap_11.md
[ok] cap_12.md
[ok] cap_13.md
[ok] cap_14.md
[ok] cap_15.md
[ok] cap_16.md
[skip anexo/apêndice] cap_17: ANEXO I - Quadro de Ênfases, Vagas e Cadastro Esperado
[skip anexo/apêndice] cap_18: ANEXO II - Cargo, Ênfases, Requisitos, Síntese das Atribuições e Remuneração
[skip anexo/apêndice] cap_19: ANEXO III - Objetos de Avaliação
[skip anexo/apêndice] cap_20: ANEXO IV - Cro

In [ ]:
def get_toc_md(path_pdf: str, provider: str = "anthropic", modelo: str = "sonnet", max_tokens: int = 32000) -> str:
    """Extrai, processa e retorna TOC formatado do PDF."""
    path_pdf = Path(path_pdf)
    
    # Carrega PDF em base64
    pdf_b64 = _carregar_pdf_base64(str(path_pdf))
    
    # Inicializa LLM
    if provider == "anthropic":
        llm = ChatAnthropic(model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0)
        bloco_pdf = {
            "type": "document",
            "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_b64},
        }
    else:
        llm = ChatGoogleGenerativeAI(model=MODELOS[provider][modelo], max_tokens=max_tokens, temperature=0)
        bloco_pdf = {"type": "media", "mime_type": "application/pdf", "data": pdf_b64}

    # Extrai TOC bruto
    mensagem = HumanMessage(content=[bloco_pdf, {"type": "text", "text": PROMPT}])
    resposta = llm.invoke([mensagem])
    resultado = resposta.content if isinstance(resposta.content, str) else "".join(b["text"] for b in resposta.content if isinstance(b, dict) and b.get("type") == "text")

    # Processa e retorna
    ultima_pagina = _contar_paginas(str(path_pdf))
    itens = inferir_janelas(resultado, ultima_pagina=ultima_pagina)
    return formatar_toc_md(itens)


# Uso:
toc_md = get_toc_md(path_pdf)
# ou: toc_md = get_toc_md(path_pdf, modelo="haiku")
# ou: toc_md = get_toc_md(path_pdf, provider="google", modelo="flash")

In [ ]:
from pathlib import Path
from pypdf import PdfReader

def _contar_paginas(caminho: str) -> int:
    return len(PdfReader(caminho).pages)

def salvar_toc_md(path_pdf: str, toc_md: str) -> Path:
    path_pdf = Path(path_pdf)
    path_saida = path_pdf.with_name(f"{path_pdf.stem}_toc.txt")
    path_saida.write_text(toc_md, encoding="utf-8")
    return path_saida

In [ ]:
resultado = extrair_toc(path_pdf)
itens = inferir_janelas(resultado, ultima_pagina=_contar_paginas(path_pdf))
toc_md = formatar_toc_md(itens)

path_toc = salvar_toc_md(path_pdf, toc_md)

print(toc_md)
print(f"\nSalvo em: {path_toc}")